# 01 · Exploración inicial de incidentes en Ecuador

Inventario de CSV, lectura acotada de una muestra y revisión de estructura. Por el tamaño total de los datos, este notebook no carga archivos completos.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'data' / 'raw').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('No se encontró data/raw desde el directorio actual')
RAW = ROOT / 'data' / 'raw'
print('Proyecto:', ROOT)

In [ ]:
files = sorted(RAW.rglob('*.csv'))
print(f'CSV detectados: {len(files)}')
inventory = pd.DataFrame({'archivo': [p.relative_to(RAW).as_posix() for p in files]})
inventory['anio'] = inventory['archivo'].str.extract(r'data(\d{4})', expand=False)
display(inventory.groupby('anio', dropna=False).size().rename('archivos').to_frame())
if files:
    sample = pd.read_csv(files[0], sep=';', dtype=str, nrows=5, encoding='utf-8-sig')
    print('Muestra:', files[0].name, '| columnas:', sample.columns.tolist())
    display(sample)
else:
    raise FileNotFoundError(f'No hay CSV en {RAW}')

In [ ]:
counts = inventory['anio'].fillna('Año no identificado').value_counts().sort_index()
out = ROOT / 'results' / 'figures'
out.mkdir(parents=True, exist_ok=True)
ax = counts.plot(kind='bar', color='#176d63', figsize=(8, 4), title='Archivos de incidentes disponibles por año')
ax.set_xlabel('Año'); ax.set_ylabel('Archivos CSV'); plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(out / 'exploracion_archivos.png', dpi=150); plt.show()